# Project 2: Market-Basket Analysis using IMDB

**Algorithms for Massive Data**  
**Master in Data Science for Economics**  
**Universita degli Studi di Milano**

This notebook implements Project 2 using the IMDB Movies Dataset. Each movie is treated as a basket, and the actors listed in `Star1`, `Star2`, `Star3`, and `Star4` are treated as items. The objective is to identify frequent actor collaborations using the Apriori algorithm.

## Executive Summary

The project converts the IMDB movie dataset into a market-basket representation and applies a custom Apriori implementation to mine frequent actor itemsets. The analysis reports frequent individual actors, pairs, triples and quadruples under different minimum-support thresholds. Runtime is also measured on increasing dataset fractions to discuss scalability.

## 1. Global Configuration

In [ ]:
# Dataset and project configuration
DATASET_LINK = "harshitshankhdhar/imdb-dataset-of-top-1000-movies-and-tv-shows"
DATA_FILE = "imdb_top_1000.csv"
DATA_DIR = "data"

# Global flag requested by the project instructions.
# Set True to use all available rows. Set False to use the first SAMPLE_SIZE rows.
USE_FULL_DATASET = True
SAMPLE_SIZE = 500

ACTOR_COLUMNS = ["Star1", "Star2", "Star3", "Star4"]
MIN_SUPPORT = 3
SUPPORT_VALUES = [2, 3, 4, 5, 10]
SCALABILITY_FRACTIONS = [0.25, 0.50, 0.75, 1.00]
SCALABILITY_REPEATS = 3

## 2. Import Libraries

In [ ]:
import os
import zipfile
import time
import itertools
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 3. Dataset Acquisition

In [ ]:
# Kaggle download cell.
# Replace "xxxxxx" with your Kaggle username and API key while running the notebook.
# Before publishing the final GitHub version, put the strings back to "xxxxxx".

os.environ["KAGGLE_USERNAME"] = "xxxxxx"
os.environ["KAGGLE_KEY"] = "xxxxxx"

# The local-file fallback below allows the notebook to run if imdb_top_1000.csv
# has already been uploaded to Colab or placed in the repository.
data_dir = Path(DATA_DIR)
data_dir.mkdir(exist_ok=True)

local_candidates = [
    Path(DATA_FILE),
    data_dir / DATA_FILE,
    Path("/content") / DATA_FILE,
    Path("/content") / DATA_DIR / DATA_FILE,
]

if not any(path.exists() for path in local_candidates):
    !pip -q install kaggle
    !kaggle datasets download -d {DATASET_LINK} -p {DATA_DIR}
    zip_files = list(data_dir.glob("*.zip"))
    for zip_path in zip_files:
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(data_dir)

## 4. Load Dataset

In [ ]:
def locate_dataset(filename=DATA_FILE):
    candidates = [
        Path(filename),
        Path(DATA_DIR) / filename,
        Path("/content") / filename,
        Path("/content") / DATA_DIR / filename,
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(f"Could not find {filename}. Upload it or run the Kaggle download cell.")

csv_path = locate_dataset()
df = pd.read_csv(csv_path)

if not USE_FULL_DATASET:
    df = df.head(SAMPLE_SIZE).copy()

print("Dataset path:", csv_path)
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
df.head()

## 5. Initial Dataset Check

In [ ]:
print("Columns in the dataset:")
print(list(df.columns))

print("\nMissing values in actor columns:")
print(df[ACTOR_COLUMNS].isna().sum())

print("\nDuplicate movie titles:", df["Series_Title"].duplicated().sum())

## 6. Data Organisation: Movies as Baskets

In [ ]:
def normalise_actor_name(value):
    if pd.isna(value):
        return None
    text = str(value).strip()
    return text if text else None


def build_baskets(data: pd.DataFrame, actor_columns=ACTOR_COLUMNS):
    baskets = []
    movie_titles = []

    for _, row in data.iterrows():
        actors = []
        for column in actor_columns:
            actor = normalise_actor_name(row[column])
            if actor is not None:
                actors.append(actor)

        # Use a set to remove duplicate actor names within the same movie.
        basket = frozenset(actors)
        if len(basket) > 0:
            baskets.append(basket)
            movie_titles.append(row["Series_Title"])

    return baskets, movie_titles

baskets, movie_titles = build_baskets(df)
unique_actors = sorted(set().union(*baskets))

print("Number of baskets:", len(baskets))
print("Number of unique actors:", len(unique_actors))
print("Minimum basket size:", min(len(b) for b in baskets))
print("Maximum basket size:", max(len(b) for b in baskets))
print("Average basket size:", round(np.mean([len(b) for b in baskets]), 3))

list(zip(movie_titles[:5], baskets[:5]))

## 7. Apriori Algorithm

The main algorithm used here is Apriori, which belongs to the course syllabus topic on frequent itemsets and market-basket analysis. The implementation below is kept to this syllabus topic and does not add unrelated models.


Apriori uses the downward-closure property: if an itemset is frequent, all of its subsets must also be frequent. Therefore, larger candidate itemsets are generated only from itemsets that were frequent at the previous level.

In [ ]:
def generate_candidates(previous_frequent, k):
    previous_frequent = set(previous_frequent)
    previous_as_tuples = sorted(tuple(sorted(itemset)) for itemset in previous_frequent)
    candidates = set()

    for i in range(len(previous_as_tuples)):
        for j in range(i + 1, len(previous_as_tuples)):
            union = frozenset(previous_as_tuples[i]).union(previous_as_tuples[j])

            if len(union) == k:
                all_subsets_frequent = all(
                    frozenset(subset) in previous_frequent
                    for subset in itertools.combinations(union, k - 1)
                )
                if all_subsets_frequent:
                    candidates.add(union)

    return candidates


def count_candidates(baskets, candidates, k):
    candidates = set(candidates)
    counts = Counter()

    for basket in baskets:
        if len(basket) >= k:
            for combination in itertools.combinations(sorted(basket), k):
                itemset = frozenset(combination)
                if itemset in candidates:
                    counts[itemset] += 1

    return counts


def apriori(baskets, min_support=MIN_SUPPORT, max_k=None):
    if max_k is None:
        max_k = max(len(basket) for basket in baskets)

    singleton_counts = Counter()
    for basket in baskets:
        for actor in basket:
            singleton_counts[frozenset([actor])] += 1

    frequent_itemsets = {
        1: Counter({itemset: count for itemset, count in singleton_counts.items() if count >= min_support})
    }

    previous_frequent = set(frequent_itemsets[1])

    for k in range(2, max_k + 1):
        candidates = generate_candidates(previous_frequent, k)
        if not candidates:
            frequent_itemsets[k] = Counter()
            continue

        candidate_counts = count_candidates(baskets, candidates, k)
        frequent_itemsets[k] = Counter({
            itemset: count for itemset, count in candidate_counts.items()
            if count >= min_support
        })

        previous_frequent = set(frequent_itemsets[k])
        if not previous_frequent:
            for remaining_k in range(k + 1, max_k + 1):
                frequent_itemsets[remaining_k] = Counter()
            break

    for k in range(1, max_k + 1):
        frequent_itemsets.setdefault(k, Counter())

    return frequent_itemsets

## 8. Main Frequent Itemset Results

In [ ]:
def itemset_to_string(itemset):
    return " + ".join(sorted(itemset))


def top_itemsets_table(frequent_itemsets, k, top_n=10):
    rows = []
    for itemset, count in frequent_itemsets.get(k, Counter()).most_common(top_n):
        rows.append({
            "Itemset size": k,
            "Actors": itemset_to_string(itemset),
            "Support count": count,
            "Support proportion": count / len(baskets)
        })
    return pd.DataFrame(rows)

frequent_itemsets = apriori(baskets, min_support=MIN_SUPPORT)

summary = pd.DataFrame({
    "Itemset size": [1, 2, 3, 4],
    "Frequent itemsets": [len(frequent_itemsets.get(k, {})) for k in [1, 2, 3, 4]]
})

summary

In [ ]:
top_itemsets_table(frequent_itemsets, k=1, top_n=10)

In [ ]:
top_itemsets_table(frequent_itemsets, k=2, top_n=10)

In [ ]:
top_itemsets_table(frequent_itemsets, k=3, top_n=10)

## 9. Support Threshold Experiment

In [ ]:
experiment_rows = []

for support in SUPPORT_VALUES:
    start_time = time.perf_counter()
    result = apriori(baskets, min_support=support)
    runtime = time.perf_counter() - start_time

    experiment_rows.append({
        "Minimum support": support,
        "Frequent actors": len(result.get(1, {})),
        "Frequent pairs": len(result.get(2, {})),
        "Frequent triples": len(result.get(3, {})),
        "Frequent quadruples": len(result.get(4, {})),
        "Runtime seconds": runtime
    })

support_experiment = pd.DataFrame(experiment_rows)
support_experiment

In [ ]:
plot_data = support_experiment.set_index("Minimum support")[[
    "Frequent actors", "Frequent pairs", "Frequent triples", "Frequent quadruples"
]]

ax = plot_data.plot(kind="bar", figsize=(10, 5))
ax.set_title("Frequent itemsets by minimum support")
ax.set_xlabel("Minimum support")
ax.set_ylabel("Number of frequent itemsets")
plt.tight_layout()
plt.show()

## 10. Scalability Experiment

In [ ]:
def time_apriori_on_fraction(baskets, fraction, min_support=MIN_SUPPORT, repeats=SCALABILITY_REPEATS):
    n = max(1, int(len(baskets) * fraction))
    subset = baskets[:n]
    runtimes = []

    for _ in range(repeats):
        start_time = time.perf_counter()
        apriori(subset, min_support=min_support)
        runtimes.append(time.perf_counter() - start_time)

    result = apriori(subset, min_support=min_support)
    total_frequent_itemsets = sum(len(result.get(k, {})) for k in result)

    return {
        "Dataset fraction": fraction,
        "Baskets": n,
        "Unique actors": len(set().union(*subset)),
        "Frequent itemsets": total_frequent_itemsets,
        "Mean runtime seconds": float(np.mean(runtimes)),
        "Std runtime seconds": float(np.std(runtimes))
    }

scalability_rows = [
    time_apriori_on_fraction(baskets, fraction)
    for fraction in SCALABILITY_FRACTIONS
]

scalability_experiment = pd.DataFrame(scalability_rows)
scalability_experiment

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar(
    scalability_experiment["Baskets"],
    scalability_experiment["Mean runtime seconds"],
    yerr=scalability_experiment["Std runtime seconds"],
    marker="o",
    capsize=4
)
ax.set_title("Apriori runtime by dataset size")
ax.set_xlabel("Number of movie baskets")
ax.set_ylabel("Mean runtime in seconds")
plt.tight_layout()
plt.show()

## 11. Result Interpretation

At minimum support 3, frequent single actors are common, frequent actor pairs are much fewer, and frequent triples are rare. This is expected because every movie basket contains at most four listed actors. Increasing the minimum support sharply reduces the number of itemsets, which is the expected behaviour of Apriori candidate pruning.

The most frequent triples correspond to recurring film franchises, such as Harry Potter, The Lord of the Rings and Star Wars. These results show that frequent itemset mining can detect repeated co-star structures without using movie titles, genres or franchise labels directly.

## 12. Conclusions

This project implemented market-basket analysis on the IMDB Movies Dataset. The custom Apriori algorithm identified recurring actor collaborations from the `Star1` to `Star4` columns. The experiments showed that lower support thresholds reveal many actor groups, while higher thresholds retain only the most repeated collaborations. The scalability experiment confirms that the same implementation can be run on increasing dataset sizes, with runtime depending on the number of baskets, basket size and generated candidates.